# Flat versus non-flat $\Lambda$CDM

This notebook uses the same particle-swarm implementation to compare a two-parameter flat $\Lambda$CDM fit with a three-parameter non-flat model. Model complexity is summarized with AIC and BIC,

$$\mathrm{AIC}=\chi^2_{\min}+2k,\qquad
\mathrm{BIC}=\chi^2_{\min}+k\ln N.$$

The production calculation uses 200 particles and 500 iterations for each model.

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from pso_cosmology import (
    load_cosmic_chronometers,
    h_flat_lcdm,
    h_nonflat_lcdm,
    h_flat_xcdm,
    h_nonflat_xcdm,
    run_pso,
    chi_square,
    model_statistics,
)

DATA_FILE = ROOT / "data" / "cosmic_chronometers_31.txt"
OUTPUT_DIR = Path(os.environ.get("PSO_OUTPUT_DIR", ROOT / "results" / "figures"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
QUICK = os.environ.get("PSO_QUICK", "0") == "1"

z, H_obs, sigma_H = load_cosmic_chronometers(DATA_FILE)
print(f"Loaded {len(z)} cosmic-chronometer measurements spanning z={z.min():.3f}–{z.max():.3f}.")
print("Quick mode:", QUICK)


In [ ]:
def objective_flat(theta):
    H0, omega_m = theta
    if not (50.0 <= H0 <= 100.0 and 0.1 <= omega_m <= 0.9):
        return np.inf
    return chi_square(h_flat_lcdm(z, H0, omega_m), H_obs, sigma_H)

def objective_nonflat(theta):
    H0, omega_m, omega_k = theta
    if not (50.0 <= H0 <= 100.0 and 0.01 <= omega_m <= 1.0 and -0.5 <= omega_k <= 0.5):
        return np.inf
    omega_lambda = 1.0 - omega_m - omega_k
    if not (0.0 <= omega_lambda <= 2.0):
        return np.inf
    return chi_square(h_nonflat_lcdm(z, H0, omega_m, omega_k), H_obs, sigma_H)

n_particles = 55 if QUICK else 200
n_iter = 120 if QUICK else 500
common = dict(n_particles=n_particles, n_iter=n_iter, inertia=0.72, cognitive=1.49, social=1.49, seed=42)
flat = run_pso(objective_flat, [(50,100),(0.1,0.9)], **common)
nonflat = run_pso(objective_nonflat, [(50,100),(0.01,1.0),(-0.5,0.5)], **common)

H0_f, om_f = flat.best_position
H0_nf, om_nf, ok_nf = nonflat.best_position
s_f = model_statistics(flat.best_score, len(z), 2)
s_nf = model_statistics(nonflat.best_score, len(z), 3)

print("Flat LCDM")
print(f"  H0={H0_f:.6f}, Omega_m={om_f:.6f}, chi2={s_f.chi2:.6f}, AIC={s_f.aic:.6f}, BIC={s_f.bic:.6f}")
print("Non-flat LCDM")
print(f"  H0={H0_nf:.6f}, Omega_m={om_nf:.6f}, Omega_k={ok_nf:.6f}, chi2={s_nf.chi2:.6f}, AIC={s_nf.aic:.6f}, BIC={s_nf.bic:.6f}")
print(f"Delta AIC={s_nf.aic-s_f.aic:.6f}; Delta BIC={s_nf.bic-s_f.bic:.6f}")

In [ ]:
z_grid = np.linspace(0.0, 2.05, 500)
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.errorbar(z, H_obs, yerr=sigma_H, fmt="o", capsize=3, alpha=0.8, label="Cosmic chronometers")
ax.plot(z_grid, h_flat_lcdm(z_grid, H0_f, om_f), lw=2.4, label=r"Flat $\Lambda$CDM")
ax.plot(z_grid, h_nonflat_lcdm(z_grid, H0_nf, om_nf, ok_nf), lw=2.4, ls="--", label=r"Non-flat $\Lambda$CDM")
ax.set(xlabel="Redshift z", ylabel=r"$H(z)$ [km s$^{-1}$ Mpc$^{-1}$]",
       title=r"Flat and non-flat $\Lambda$CDM best fits")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "flat_vs_nonflat_best_fit.png", dpi=220)
plt.show()

In [ ]:
labels = ["Flat $\\Lambda$CDM", "Non-flat $\\Lambda$CDM"]
chi2_values = [s_f.chi2, s_nf.chi2]
aic_values = [s_f.aic, s_nf.aic]
bic_values = [s_f.bic, s_nf.bic]
x = np.arange(len(labels)); width = 0.24
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.bar(x-width, chi2_values, width, label=r"$\chi^2_{\min}$")
ax.bar(x, aic_values, width, label="AIC")
ax.bar(x+width, bic_values, width, label="BIC")
ax.set_xticks(x, labels)
ax.set_ylabel("Statistic")
ax.set_title("Model-selection comparison")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "flat_vs_nonflat_model_selection.png", dpi=220)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(flat.best_score_history, lw=2.2, label=r"Flat $\Lambda$CDM")
ax.plot(nonflat.best_score_history, lw=2.2, label=r"Non-flat $\Lambda$CDM")
ax.set(xlabel="PSO iteration", ylabel=r"Best $\chi^2$ found so far", title="PSO convergence")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "flat_vs_nonflat_convergence.png", dpi=220)
plt.show()